# 03 — PCA exploratória: MCF10A sob três condições no E2

A PCA é uma técnica não supervisionada de redução de dimensionalidade, **não um classificador**. Aqui aplicamos a mesma preparação de dados elétrica sem olhar para os rótulos na etapa de ajuste.

- **12 aplicações físicas** de VEs (4 `Controle1`, 4 `Controle2` e 4 `cond.HCC`) + PBS local.
- O espectro `P05_eletrodo2(1).txt` com aproximadamente 1,45 Ω é excluído da análise porque é incompatível com as três outras leituras da mesma aplicação PBS.
- Três leituras da mesma gota são agregadas: **não são três réplicas independentes**.
- O experimento E2 não se mistura à classificação E7.
- Sem imagens salvas automaticamente.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import LeaveOneGroupOut, LeaveOneOut
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from modelos_basicos import basic_models, cv_predictions
plt.rcParams.update({'figure.figsize':(9,5),'font.size':11,'axes.spines.top':False,'axes.spines.right':False})

from eis_utils import read_all, get_e02_pair_table, band_average
aplicacoes=read_all(Path('dados/Medicoes_EIS_Por_Dispositivo_ATUALIZADO_E02.zip'),drop_known_short=True)
e2=get_e02_pair_table(aplicacoes)
assert len(e2)==12 and set(e2.condition.value_counts())=={4}
print(e2[['order','condition','application','scans_ves','scans_pbs']].to_string(index=False))

## Curvas elétricas das 12 aplicações, normalizadas pelo respectivo PBS

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12.8,4.8))
for cond,g in e2.groupby('condition'):
    for k,(_,r) in enumerate(g.iterrows()):
        axes[0].plot(r.f,r.phase_delta,alpha=.6,lw=1.4,label=cond if k==0 else '_nolegend_')
        axes[1].plot(r.f,r.logratio_z,alpha=.6,lw=1.4,label=cond if k==0 else '_nolegend_')
axes[0].set(xscale='log',xlabel='Frequência (Hz)',ylabel='Δ(−Fase) (°)',title='Diferença de fase: VEs − PBS')
axes[1].set(xscale='log',xlabel='Frequência (Hz)',ylabel='log₁₀ (|Z| VEs / PBS)',title='Razão de impedância: VEs / PBS')
for ax in axes: ax.grid(alpha=.17);ax.axhline(0,color='gray',lw=.8)
axes[0].legend();plt.tight_layout();plt.show()

## PCA de sete bandas de fase, com anotação da ordem de aplicação

In [ ]:
EDGES=np.logspace(-1,6,8)
X=np.vstack([[band_average(r.f,r.phase_delta,lo,hi) for lo,hi in zip(EDGES[:-1],EDGES[1:])]
             for _,r in e2.iterrows()])
assert X.shape==(12,7) and np.isfinite(X).all()
Xscale=StandardScaler().fit_transform(X)
pca=PCA(n_components=2); pcs=pca.fit_transform(Xscale)
fig,ax=plt.subplots(figsize=(8.7,5.6))
for cond,g in e2.groupby('condition'):
    idx=g.index.to_numpy();ax.scatter(pcs[idx,0],pcs[idx,1],s=85,alpha=.85,label=cond)
    for i in idx:ax.annotate(str(e2.loc[i,'order']),pcs[i],xytext=(4,5),textcoords='offset points',fontsize=8)
ax.set(xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',title='E2: PCA da fase normalizada pelo PBS')
ax.axhline(0,color='gray',lw=.8);ax.axvline(0,color='gray',lw=.8);ax.legend(title='Condição');ax.grid(alpha=.14)
plt.tight_layout();plt.show()

## Diagnóstico de deriva: colorir a MESMA PCA pela ordem, não pela condição

In [ ]:
fig,ax=plt.subplots(figsize=(8.7,5.5))
s=ax.scatter(pcs[:,0],pcs[:,1],c=e2.order.to_numpy(),cmap='viridis',s=95)
for i,r in e2.iterrows():ax.annotate(str(r.order),pcs[i],xytext=(4,4),textcoords='offset points',fontsize=9)
ax.set(xlabel='PC1',ylabel='PC2',title='E2: mesmos scores, coloridos pela ordem experimental')
fig.colorbar(s,ax=ax,label='Ordem da gota (1–12)')
ax.grid(alpha=.14);plt.tight_layout();plt.show()
print('Correlação Spearman (ordem, PC1):',round(pd.Series(e2.order).corr(pd.Series(pcs[:,0]),method='spearman'),3))
print('Correlação Spearman (ordem, PC2):',round(pd.Series(e2.order).corr(pd.Series(pcs[:,1]),method='spearman'),3))

## Interpretação

Uma separação aparente de grupos na PCA **não prova** transformação fenotípica, especificidade da resposta elétrica às VEs nem reprodutibilidade entre lotes biológicos. As quatro gotas por condição são aplicações técnicas de uma preparação, e o meio condicionado pode conter componentes pré-existentes da HCC1806. Descrever o resultado como estudo **exploratório das preparações**.